In [ ]:
# ================================================================
# PHYSICAL AI DEMO
# Click a target on a room image and watch the robot move toward it
#
# Google Colab
# ================================================================

!pip -q install pillow

import cv2
import base64
import numpy as np

from PIL import Image as PILImage
from PIL import ImageDraw
from IPython.display import display, Image

from google.colab import files
from google.colab.output import eval_js


# ================================================================
# STEP 1 - UPLOAD ROOM IMAGE
# ================================================================

print("Upload your room.jpg image")

uploaded = files.upload()

image_path = next(iter(uploaded))

print("Uploaded:", image_path)


# ================================================================
# STEP 2 - READ IMAGE
# ================================================================

room = cv2.imread(image_path)

room = cv2.cvtColor(
    room,
    cv2.COLOR_BGR2RGB
)

height, width, _ = room.shape

print("Image size:", width, "x", height)


# ================================================================
# STEP 3 - CONVERT IMAGE TO BASE64
# This allows the image to be displayed inside browser JavaScript
# ================================================================

_, buffer = cv2.imencode(
    ".jpg",
    cv2.cvtColor(room, cv2.COLOR_RGB2BGR)
)

image_base64 = base64.b64encode(
    buffer
).decode()


# ================================================================
# STEP 4 - CLICK ON IMAGE TO SELECT TARGET
# ================================================================

javascript = f"""
new Promise((resolve) => {{

    const container = document.createElement('div');

    container.innerHTML = `
        <h3>Click anywhere on the image to choose the ROBOT TARGET</h3>
        <canvas id="targetCanvas"
                style="border:3px solid black; cursor:crosshair;">
        </canvas>
        <p id="status">
            Waiting for target selection...
        </p>
    `;

    document.body.appendChild(container);


    const canvas =
        document.getElementById("targetCanvas");

    const ctx =
        canvas.getContext("2d");


    const img = new Image();

    img.src =
        "data:image/jpeg;base64,{image_base64}";


    img.onload = function() {{

        // Limit displayed image size
        const maxWidth = 900;

        let scale = 1;

        if (img.width > maxWidth) {{
            scale = maxWidth / img.width;
        }}

        canvas.width =
            img.width * scale;

        canvas.height =
            img.height * scale;

        ctx.drawImage(
            img,
            0,
            0,
            canvas.width,
            canvas.height
        );


        canvas.onclick = function(event) {{

            const rect =
                canvas.getBoundingClientRect();


            const clickX =
                event.clientX - rect.left;

            const clickY =
                event.clientY - rect.top;


            // Convert displayed coordinates
            // back to original image coordinates

            const originalX =
                clickX / scale;

            const originalY =
                clickY / scale;


            // Draw target circle

            ctx.beginPath();

            ctx.arc(
                clickX,
                clickY,
                18,
                0,
                2 * Math.PI
            );

            ctx.strokeStyle =
                "red";

            ctx.lineWidth =
                6;

            ctx.stroke();


            ctx.font =
                "20px Arial";

            ctx.fillStyle =
                "red";

            ctx.fillText(
                "TARGET",
                clickX + 20,
                clickY
            );


            document.getElementById(
                "status"
            ).innerHTML =
                "Target selected";


            resolve([
                originalX,
                originalY
            ]);
        }};
    }};
}})
"""


target_coordinates = eval_js(
    javascript
)


target = np.array(
    target_coordinates,
    dtype=float
)


print("\nTARGET SELECTED")

print(
    "Target X:",
    int(target[0])
)

print(
    "Target Y:",
    int(target[1])
)


# ================================================================
# STEP 5 - DEFINE ROBOT START POSITION
#
# Robot starts near bottom-left of room
# ================================================================

start = np.array(
    [
        width * 0.10,
        height * 0.82
    ],
    dtype=float
)


print("\nRobot Start:")

print(
    int(start[0]),
    int(start[1])
)


# ================================================================
# STEP 6 - CREATE ROBOT PATH
# ================================================================

NUMBER_OF_FRAMES = 70


trajectory = []


for t in np.linspace(
    0,
    1,
    NUMBER_OF_FRAMES
):

    # Linear path

    position = (

        start * (1 - t)

        +

        target * t

    )


    trajectory.append(
        position.copy()
    )


trajectory = np.array(
    trajectory
)


# ================================================================
# STEP 7 - CREATE ANIMATION FRAMES
# ================================================================

frames = []


for frame_number, position in enumerate(
    trajectory
):

    frame = PILImage.fromarray(
        room.copy()
    )


    draw = ImageDraw.Draw(
        frame
    )


    # Current robot position

    x = int(
        position[0]
    )

    y = int(
        position[1]
    )


    # Target position

    tx = int(
        target[0]
    )

    ty = int(
        target[1]
    )


    # ============================================================
    # DRAW TARGET
    # ============================================================

    draw.ellipse(

        [
            tx - 25,
            ty - 25,
            tx + 25,
            ty + 25
        ],

        outline="red",

        width=7
    )


    draw.text(

        (
            tx + 30,
            ty - 10
        ),

        "TARGET",

        fill="red"
    )


    # ============================================================
    # DRAW PATH ALREADY TRAVELLED
    # ============================================================

    if frame_number > 1:

        path_points = [

            (
                int(p[0]),
                int(p[1])
            )

            for p in trajectory[
                :frame_number + 1
            ]

        ]


        draw.line(

            path_points,

            fill="yellow",

            width=6

        )


    # ============================================================
    # DRAW ROBOT BODY
    # ============================================================

    robot_width = 60

    robot_height = 65


    draw.rounded_rectangle(

        [
            x - robot_width//2,
            y - robot_height//2,

            x + robot_width//2,
            y + robot_height//2
        ],

        radius=12,

        fill="blue",

        outline="white",

        width=4

    )


    # ============================================================
    # ROBOT HEAD
    # ============================================================

    draw.rounded_rectangle(

        [
            x - 23,
            y - 65,

            x + 23,
            y - 32
        ],

        radius=6,

        fill="lightblue",

        outline="white",

        width=3

    )


    # ============================================================
    # EYES
    # ============================================================

    draw.ellipse(

        [
            x - 14,
            y - 55,

            x - 7,
            y - 48
        ],

        fill="black"

    )


    draw.ellipse(

        [
            x + 7,
            y - 55,

            x + 14,
            y - 48
        ],

        fill="black"

    )


    # ============================================================
    # WHEELS
    # ============================================================

    draw.ellipse(

        [
            x - 38,
            y + 20,

            x - 20,
            y + 45
        ],

        fill="black"

    )


    draw.ellipse(

        [
            x + 20,
            y + 20,

            x + 38,
            y + 45
        ],

        fill="black"

    )


    # ============================================================
    # STATUS MESSAGE
    # ============================================================

    draw.text(

        (
            20,
            20
        ),

        f"Physical AI Robot - Step {frame_number}",

        fill="white",

        stroke_width=2,

        stroke_fill="black"

    )


    frames.append(
        frame
    )


# ================================================================
# STEP 8 - SAVE ANIMATED GIF
# ================================================================

output_file = (
    "/content/"
    "robot_moves_to_selected_target.gif"
)


frames[0].save(

    output_file,

    save_all=True,

    append_images=frames[1:],

    duration=120,

    loop=0

)


print(
    "\nAnimation created successfully!"
)


# ================================================================
# STEP 9 - DISPLAY ROBOT MOVEMENT
# ================================================================

display(

    Image(
        filename=output_file
    )

)